In [42]:
!pip install feedparser

In [43]:
#All the necessary libraries.
import requests
import pandas as pd
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from dateutil import parser as dateutil_parser
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import feedparser


In [44]:
#All the scmema columns as per mentioned in the Build Test
SCHEMA_COLUMNS = [
    "event_datetime_utc",
    "source_name",
    "source_url",
    "source_type",
    "claim_text",
    "country",
    "location_text",
    "actor_1",
    "actor_2",
    "event_type",
    "domain",
    "severity_score",
    "confidence_score",
    "information_status",
    "tags",
    "last_updated_at"
]

In [45]:
# to_utc is defined below with proper multi-format support (see cell 8)
# This cell is intentionally left as a placeholder.


In [46]:
#Event Classification
def classify_event(text):
    text = text.lower()

    if "missile" in text:
        return "missile_attack"
    elif "airstrike" in text or "strike" in text:
        return "airstrike"
    elif "troop" in text:
        return "troop_movement"
    elif "protest" in text:
        return "protest"
    elif "warning" in text or "threat" in text:
        return "political_statement"
    else:
        return "other"

In [47]:
GEO_MAP = {
    "iran": ["tehran", "shiraz", "isfahan", "tabriz", "mashhad", "bushehr", "bandar abbas", "kermanshah"],
    "israel": ["tel aviv", "jerusalem", "haifa", "ashdod", "beersheba", "eilat", "lod", "ramat gan", "gaza"],
    "lebanon": ["beirut", "tyre", "sidon", "tripoli", "nabatieh"],
    "iraq": ["baghdad", "erbil", "basra", "mosul"],
    "yemen": ["sanaa", "hodeidah", "aden"]
}

# def geocode_event(text):
#     text = text.lower()
#     # Check for specific cities first (more precise)
#     for country, cities in GEO_MAP.items():
#         for city in cities:
#             if city in text:
#                 return country.capitalize(), city.capitalize()

#     # If no city, check for country names
#     for country in GEO_MAP.keys():
#         if country in text:
#             return country.capitalize(), None

#     return "WestAsia", None

In [48]:
def geocode_event(text):
    text = text.lower()

    # Check cities first
    for country, cities in GEO_MAP.items():
        for city in cities:
            if city in text:
                return country.capitalize(), city.capitalize()

    # If no city, check country
    for country in GEO_MAP.keys():
        if country in text:
            return country.capitalize(), country.capitalize()   # ✅ FIX

    return "WestAsia", "West Asia"   # optional fallback

In [49]:
#Actor Extraction
def extract_actors(text):
    text = text.lower()
    actor_1, actor_2 = None, None

    # Mapping keywords to Standardized Names
    mapping = {
        "iran": "Iran",
        "israel": "Israel",
        "us": "USA",
        "united states": "USA",
        "gcc": "GCC",
        "gulf cooperation": "GCC",
        "un ": "UN", # space to avoid matching 'under'
        "united nations": "UN",
        "eu ": "EU",
        "european union": "EU",
        "hezbollah": "Hezbollah",
        "hamas": "Hamas",
        "houthis": "Houthis",
        "russia": "Russia",
        "china": "China"
    }

    found_actors = [name for key, name in mapping.items() if key in text]

    if len(found_actors) >= 1:
      actor_1 = found_actors[0]
    else:
      actor_1=None

    if len(found_actors) >= 2:
       actor_2 = found_actors[1]
    else:
      actor_2=None

    return actor_1, actor_2

In [50]:
from urllib.parse import urlparse

def get_source_type(raw):
    # --- 0. Extract source name safely ---
    source_name = ""

    if isinstance(raw.get("source"), dict):
        source_name = raw.get("source", {}).get("title", "").lower()
    else:
        source_name = (raw.get("title") or "").lower()

    # --- 0.1 Extract domain from URL ---
    url = raw.get("link", "")
    domain = urlparse(url).netloc.lower() if url else ""

    # --- 1. Custom mapping (HIGHEST PRIORITY) ---
    if (
        "isw" in source_name or
        "institute for the study of war" in source_name or
        "understandingwar.org" in domain
    ):
        return "osint"

    # --- 2. Default ---
    return "news"

In [51]:
def classify_domain(text):
    text = text.lower()

    # --- Military ---
    if any(w in text for w in [
        "strike", "missile", "attack", "airstrike",
        "drone", "clash", "explosion", "raid"
    ]):
        return "military"

    # --- Political ---
    if any(w in text for w in [
        "talks", "meeting", "agreement", "ceasefire",
        "negotiation", "diplomacy", "warns", "statement","Report"
    ]):
        return "political"

    # --- Economic ---
    if any(w in text for w in [
        "sanction", "trade", "oil", "gas", "economy","Special Report"
    ]):
        return "economic"

    # --- Humanitarian ---
    if any(w in text for w in [
        "civilian", "aid", "refugee", "injured", "killed", "dead"
    ]):
        return "humanitarian"

    # --- Cyber ---
    if any(w in text for w in [
        "cyber", "hack", "malware"
    ]):
        return "cyber"

    return "other"

In [52]:
id="tag_extractor_basic"
def extract_tags(text):
    text = text.lower()

    TAG_KEYWORDS = [
        # Locations
        "iran", "israel", "gaza", "lebanon", "tehran", "tel aviv",

        # Weapons / actions
        "missile", "drone", "airstrike", "strike", "attack", "explosion",

        # Political
        "ceasefire", "talks", "sanctions", "war", "conflict",

        # Actors
        "hamas", "hezbollah", "idf"
    ]

    tags = [word for word in TAG_KEYWORDS if word in text]

    return list(set(tags))  # remove duplicates

In [53]:
from datetime import datetime, timezone

In [54]:
# Source credibility tiers based on editorial standards and verification methodology
SOURCE_CONFIDENCE = {
    # Tier 1 – Primary research / think-tank with on-ground verification
    "ISW":          0.90,
    # Tier 2 – Established international wire/broadcast with editorial oversight
    "BBC":          0.80,
    "Reuters":      0.80,
    # Tier 3 – Aggregators / event databases (machine-curated, less editorial review)
    "GDELT":        0.60,
    # Tier 4 – Google News aggregation (source quality varies widely)
    "Google News":  0.50,
}
DEFAULT_CONFIDENCE = 0.40  # Unknown or unlisted source

def compute_confidence(source, text):
    """Base confidence from source tier, then adjust for claim-level signals."""
    score = SOURCE_CONFIDENCE.get(source, DEFAULT_CONFIDENCE)

    text_l = text.lower()

    # Boost: corroborating language signals higher certainty
    if any(w in text_l for w in ["confirmed", "official", "ministry", "pentagon", "idf"]):
        score = min(score + 0.10, 1.0)

    # Penalty: hedging / unverified language
    if any(w in text_l for w in ["report", "claim", "alleged", "unconfirmed", "sources say"]):
        score = max(score - 0.10, 0.10)

    return round(score, 2)

def normalize_event(raw):
    text = raw.get("title", "") or ""
    source = raw.get("source", "Unknown Source")

    country, location = geocode_event(text)

    actor_1, actor_2 = extract_actors(text)
    confidence = compute_confidence(source, text)

    return {
        "event_datetime_utc": to_utc(raw.get("pub_date"), source),
        "source_name": source,
        "source_url": raw.get("link"),
        "source_type": get_source_type(raw),
        "claim_text": text,
        "country": country,
        "location_text": location,
        "actor_1": actor_1,
        "actor_2": actor_2,
        "event_type": classify_event(text),
        "domain": classify_domain(text),
        "severity_score": None,
        "confidence_score": confidence,
        "information_status": "verified" if confidence >= 0.75 else "unverified",
        "tags": extract_tags(text),
        "last_updated_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    }


In [55]:
def to_utc(date_str, source="unknown"):
    """Robust UTC converter using dateutil.parser."""
    if not date_str or date_str == "Special Report":
        return None
    try:
        dt = dateutil_parser.parse(date_str)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception as e:
        print(f"Date error for {source}: {e}")
        return None


In [56]:
#News Ingestion
def fetch_news():
    url = "https://news.google.com/rss/search?q=Iran+Israel+conflict&hl=en-IN&gl=IN&ceid=IN:en"

    response = requests.get(url)
    return response.text

In [57]:
def parse_rss(xml_data):
    root = ET.fromstring(xml_data)
    items = root.findall(".//item")

    events = []

    for item in items:  # removed limit
        title = item.find("title").text
        link = item.find("link").text
        pub_date = item.find("pubDate").text

        events.append({
            "title": title,
            "link": link,
            "pub_date": pub_date,
            "source": "Google News"
        })

    return events

In [58]:
xml_data = fetch_news()
raw_events = parse_rss(xml_data)

normalized_events = [normalize_event(e) for e in raw_events]

df = pd.DataFrame(normalized_events)

df.to_json("data_processed_events.json", orient="records", indent=2)


In [59]:
!pip install feedparser

import feedparser

def fetch_reuters():
    url = "https://www.reutersagency.com/feed/?best-topics=middle-east"
    feed = feedparser.parse(url)
    return feed


In [60]:
from curl_cffi import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

def fetch_isw_data():
    url = "https://www.understandingwar.org/publications"
    if "understandingwar.org" in url:
      source_name = "ISW"

    try:
        # Using chrome110 impersonation to bypass the 403 Forbidden wall
        response = requests.get(url, impersonate="chrome110", timeout=30)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Connection Error: {e}")
        return None

def parse_isw_updates(html):
    if not html:
        return []
    soup = BeautifulSoup(html, "html.parser")
    articles = []

    # This selector targets the specific titles found in your successful fallback test
    # It looks for links inside 'views-field-title' divs or 'h3' headers
    targets = soup.select(".views-field-title a, h3 a")

    for link_tag in targets:
        title = link_tag.get_text(strip=True)
        link = urljoin("https://www.understandingwar.org", link_tag['href'])

        # We only want research/report links, not site navigation
        if "/research/" in link or "/backgrounder/" in link:
            # Simple RegEx to pull the date out of the title if it exists
            # Example: "March 25, 2026"
            date_match = re.search(r'[A-Z][a-z]+ \d{1,2}, \d{4}', title)
            date = date_match.group(0) if date_match else "Special Report"

            articles.append({
                "title": title,
                "link": link,
                "pub_date": date,
                "source": "ISW"
            })

    # Remove duplicates (sometimes the same link appears in multiple tags)
    seen = set()
    unique_articles = []
    for a in articles:
        if a['link'] not in seen:
            unique_articles.append(a)
            seen.add(a['link'])

    return unique_articles[:100] # Return the 10 most recent

# Run it
html = fetch_isw_data()
if html:
    isw_updates = parse_isw_updates(html)
    print(f"Successfully captured {len(isw_updates)} updates:\n")
    for item in isw_updates:
        print(f"[{item['pub_date']}] {item['title']}")
        print(f"URL: {item['link']}\n")


Successfully captured 30 updates:

[March 26, 2026] Russian Occupation Update, March 26, 2026
URL: https://understandingwar.org/research/russia-ukraine/russian-occupation-update-march-26-2026/

[March 25, 2026] Hezbollah-Claimed Attacks Targeting IDF Forces and Positions in Israel Between March 1 and March 25, 2026
URL: https://understandingwar.org/research/middle-east/hezbollah-claimed-attacks-targeting-idf-forces-and-positions-in-israel-between-march-1-and-march-25-2026/

[March 25, 2026] Russian Offensive Campaign Assessment, March 25, 2026
URL: https://understandingwar.org/research/russia-ukraine/russian-offensive-campaign-assessment-march-25-2026/

[March 25, 2026] Iran Update Special Report, March 25, 2026
URL: https://understandingwar.org/research/middle-east/iran-update-special-report-march-25-2026/

[March 25, 2026] Korean Peninsula Update, March 25, 2026
URL: https://understandingwar.org/research/china-taiwan/korean-peninsula-update-march-25-2026/

[March 24, 2026] Iran Updat

In [61]:
def fetch_gdelt():
    url = "https://api.gdeltproject.org/api/v2/doc/doc?query=Iran&mode=ArtList&maxrecords=10&format=json"

    try:
        response = requests.get(url, timeout=30)  # ⬅️ increased timeout

        if response.status_code != 200:
            print("GDELT request failed:", response.status_code)
            return {}

        return response.json()

    except Exception as e:
        print("GDELT error:", e)
        return {}

def parse_gdelt(data):
    events = []

    if not data:
        return events

    articles = data.get("articles", [])

    for art in articles:
        events.append({
            "title": art.get("title"),
            "link": art.get("url"),
            "pub_date": art.get("seendate"),
            "source": "GDELT"
        })

    return events

In [62]:
def fetch_bbc():
    url = "http://feeds.bbci.co.uk/news/world/rss.xml"
    response = requests.get(url)
    return response.text

def parse_bbc(xml_data):
    if not xml_data:
        return []
    try:
        root = ET.fromstring(xml_data)
    except ET.ParseError as e:
        print(f"BBC XML parse error: {e}")
        return []
    items = root.findall(".//item")

    events = []

    for item in items[:100]:
        title = item.find("title").text
        link = item.find("link").text
        pub_date = item.find("pubDate").text

        events.append({
            "title": title,
            "link": link,
            "pub_date": pub_date,
            "source": "BBC"
        })

    return events


In [63]:
def deduplicate_events(events):
    seen_urls = set()
    seen_titles = set()
    unique_events = []

    for e in events:
        # Normalize title for comparison (lowercase and strip special chars)
        clean_title = re.sub(r'[^a-zA-Z0-9\s]', '', e['title'].lower()).strip()
        url = e['link']

        if url not in seen_urls and clean_title not in seen_titles:
            unique_events.append(e)
            seen_urls.add(url)
            seen_titles.add(clean_title)

    return unique_events

In [64]:
def run_analysis_layer(df):
    # Ensure chronological order
    df['event_datetime_utc'] = pd.to_datetime(df['event_datetime_utc'])
    df = df.sort_values('event_datetime_utc')

    # A. ESCALATION SIGNAL: 3-event rolling average of severity
    # If the last few events are significantly higher than the baseline, it's an anomaly.
    df['severity_ma'] = df['severity_score'].rolling(window=3).mean()
    global_avg_severity = df['severity_score'].mean()

    # Logic: Mark as escalation if rolling average is 1.5x the global average
    df['is_escalation'] = df['severity_ma'] > (global_avg_severity * 1.5)

    # B. CONTRADICTION DETECTOR: High Volatility vs. Low Confidence
    # Identifying where 'unverified' news is spiking but 'verified' (ISW) is silent
    df['source_volatility'] = df.groupby('source_name')['severity_score'].transform('std').fillna(0)

    # C. PATTERN IDENTIFICATION: Actor Engagement Ratio
    # Are events involving 'USA' increasing relative to 'Israel'?
    usa_involvement = df['claim_text'].str.contains('US|USA|Trump', case=False).rolling(window=5).sum()
    df['usa_engagement_surge'] = usa_involvement >= 3

    return df

In [65]:
def generate_executive_summary(df):
    high_sev_count = len(df[df['severity_score'] >= 8])
    recent_escalation = df['is_escalation'].iloc[-1] if not df.empty else False
    top_location = df['location_text'].mode()[0] if not df['location_text'].empty else "Multiple"

    status_color = "🔴 HIGH ALERT" if recent_escalation or high_sev_count > 5 else "🟡 MONITORING"

    summary = f"""
    ### 🛡️ Strategic Intelligence Summary
    **Current Posture:** {status_color}

    * **Escalation Signal:** {"Detected" if recent_escalation else "None"} (based on severity surge).
    * **Hotspot:** {top_location} has the highest frequency of kinetic reports.
    * **Confidence Gap:** {len(df[df['information_status'] == 'unverified'])} events are currently unverified.
    * **Key Actors:** {df['actor_1'].mode()[0]} vs {df['actor_2'].mode()[0]} remains the primary dyad.
    """
    return summary

In [66]:
# 1. Fetch all data
xml_google = fetch_news()
google_events = parse_rss(xml_google)

feed_isw = fetch_isw_data()
isw_events = parse_isw_updates(feed_isw)

xml_gdelt = fetch_gdelt()
gdelt_events = parse_gdelt(xml_gdelt)

xml_bbc = fetch_bbc()
bbc_events = parse_bbc(xml_bbc)

# --- 1. COMBINE ALL SOURCES ---
all_raw_events = google_events + isw_events + gdelt_events + bbc_events

# --- 2. RELEVANCE FILTER ---
keywords = ["iran", "israel", "war", "conflict", "strike", "missile", "tehran", "tel aviv", "trump", "ceasefire"]
filtered_raw = [
    e for e in all_raw_events
    if any(k in (e.get('title') or '').lower() for k in keywords)
]

# --- 3. DEDUPLICATION ---
unique_raw = deduplicate_events(filtered_raw)

# --- 4. NORMALIZATION ---
normalized_list = [normalize_event(e) for e in unique_raw]

# --- 5. POST-PROCESSING (Severity & Final Cleanup) ---
final_processed_events = []

for event in normalized_list:
    text = (event["claim_text"] or "").lower()
    if any(w in text for w in ["killed", "dead", "attack", "missile", "strike"]):
        event["severity_score"] = 8
    elif any(w in text for w in ["threat", "warns", "military", "border"]):
        event["severity_score"] = 5
    else:
        event["severity_score"] = 3

    if event.get("country") is not None:
        final_processed_events.append(event)

# --- 6. DATAFRAME ---
df = pd.DataFrame(final_processed_events)

# --- 7. ANALYSIS LAYER (must run AFTER df is built) ---
if not df.empty:
    df = run_analysis_layer(df)

# --- 8. EXECUTIVE SUMMARY ---
if not df.empty:
    exec_summary = generate_executive_summary(df)
    print(exec_summary)

# --- 9. EXPORT ---
if not df.empty:
    df_export = df[SCHEMA_COLUMNS] if all(c in df.columns for c in SCHEMA_COLUMNS) else df
    df_export.to_json("data_processed_events.json", orient="records", indent=2)

print(f"Total Raw: {len(all_raw_events)}")
print(f"After Cleanup & Filtering: {len(df)}")
df.head(20)


GDELT request failed: 429

    ### 🛡️ Strategic Intelligence Summary
    **Current Posture:** 🔴 HIGH ALERT

    * **Escalation Signal:** None (based on severity surge).
    * **Hotspot:** Iran has the highest frequency of kinetic reports.
    * **Confidence Gap:** 97 events are currently unverified.
    * **Key Actors:** Iran vs Israel remains the primary dyad.
    
Total Raw: 155
After Cleanup & Filtering: 117


,event_datetime_utc,source_name,source_url,source_type,claim_text,country,location_text,actor_1,actor_2,event_type,domain,severity_score,confidence_score,information_status,tags,last_updated_at,severity_ma,is_escalation,source_volatility,usa_engagement_surge
107,2026-02-28 00:00:00+00:00,ISW,https://understandingwar.org/research/middle-e...,osint,"Iranian Ballistic Missiles, Cruise Missiles, a...",Iran,Iran,Iran,None,missile_attack,military,8,0.9,verified,"[missile, drone, iran]",2026-03-26 19:04:12 UTC,NaN,False,2.288689,False
108,2026-02-28 00:00:00+00:00,ISW,https://understandingwar.org/research/middle-e...,osint,"Iranian Ballistic Missiles, Cruise Missiles, a...",Iran,Iran,Iran,None,missile_attack,military,8,0.9,verified,"[missile, drone, iran]",2026-03-26 19:04:12 UTC,NaN,False,2.288689,False
104,2026-03-01 00:00:00+00:00,ISW,https://understandingwar.org/research/middle-e...,osint,Iranian Ballistic Missiles and Drones Launched...,Iran,Iran,Iran,None,missile_attack,military,8,0.9,verified,"[missile, drone, iran]",2026-03-26 19:04:12 UTC,8.000000,True,2.288689,False
96,2026-03-04 08:00:00+00:00,Google News,https://news.google.com/rss/articles/CBMirAFBV...,news,From Allies To Adversaries: The Iran–Israel St...,Iran,Iran,Iran,Israel,other,other,3,0.5,unverified,"[israel, iran]",2026-03-26 19:04:12 UTC,6.333333,False,2.161042,False
93,2026-03-04 08:00:00+00:00,Google News,https://news.google.com/rss/articles/CBMi1AFBV...,news,Iran-Israel war: What has happened so far - Th...,Iran,Iran,Iran,Israel,other,other,3,0.5,unverified,"[israel, iran, war]",2026-03-26 19:04:12 UTC,4.666667,False,2.161042,False
91,2026-03-07 08:00:00+00:00,Google News,https://news.google.com/rss/articles/CBMi4wFBV...,news,Iran-Israel conflict heightens risks to India’...,Iran,Iran,Iran,Israel,other,other,3,0.5,unverified,"[israel, iran, conflict]",2026-03-26 19:04:12 UTC,3.000000,False,2.161042,False
95,2026-03-10 07:00:00+00:00,Google News,https://news.google.com/rss/articles/CBMinwFBV...,news,Iran war: What is happening on day 12 of US-Is...,Iran,Iran,Iran,Israel,other,military,8,0.5,unverified,"[israel, iran, attack, war]",2026-03-26 19:04:12 UTC,4.666667,False,2.161042,False
37,2026-03-13 07:00:00+00:00,Google News,https://news.google.com/rss/articles/CBMingFBV...,news,Iran’s War With Israel and the United States |...,Iran,Iran,Iran,Israel,other,other,3,0.5,unverified,"[israel, iran, conflict, war]",2026-03-26 19:04:12 UTC,4.666667,False,2.161042,False
109,2026-03-17 00:00:00+00:00,ISW,https://understandingwar.org/research/middle-e...,osint,"Iran Update Special Report, March 17, 2026",Iran,Iran,Iran,None,other,other,3,0.8,verified,[iran],2026-03-26 19:04:12 UTC,4.666667,False,2.288689,False
111,2026-03-17 00:00:00+00:00,ISW,https://understandingwar.org/research/middle-e...,osint,Hierarchy of the Iranian Armed Forces as of 8:...,Iran,Iran,Iran,None,other,other,3,0.9,verified,[iran],2026-03-26 19:04:12 UTC,3.000000,False,2.288689,False


In [67]:
df_export.to_csv("data_processed_events.csv", index=False)

In [68]:
print("Google:", len(google_events))
print("Understanding War:", len(isw_events))
print("GDELT:", len(gdelt_events))
print("BBC:", len(bbc_events))
all_raw_events = google_events + isw_events + gdelt_events + bbc_events

print("Total:", len(all_raw_events))
df["source_name"].value_counts()


Google: 100
Understanding War: 30
GDELT: 0
BBC: 25
Total: 155


,count
source_name,
Google News,97
ISW,15
BBC,5


In [69]:
#df

In [70]:
# 1. Fix the Cloudflare installation (Corrected Flags)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 2. Re-verify Streamlit is running
import os
import subprocess
import sys
!fuser -k 8501/tcp
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# 3. Launch the Tunnel
print("\n--- LOOK FOR THE 'TRYCLOUDFLARE' LINK BELOW ---")
!cloudflared tunnel --url http://localhost:8501

(Reading database ... 118198 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...

--- LOOK FOR THE 'TRYCLOUDFLARE' LINK BELOW ---
2026-03-26T19:04:14Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-26T19:04:14Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-26

In [71]:
with open('app.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import json
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import random

# 1. TACTICAL UI CONFIG
st.set_page_config(page_title="SAIG | OMNI-RECON", layout="wide", initial_sidebar_state="expanded")

# "Heavy" CSS Injection (Tactical Dark Mode)
st.markdown(\"\"\"
    <style>
    @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&display=swap');
    html, body, [class*="css"]  { font-family: 'JetBrains Mono', monospace; background-color: #050505; }
    .stMetric {
        background: linear-gradient(135deg, #111 0%, #000 100%);
        border: 1px solid #00ff41;
        box-shadow: 0px 0px 10px #00ff4144;
        border-radius: 4px; padding: 20px;
    }
    div[data-testid="stMetricValue"] { color: #00ff41; text-shadow: 0 0 5px #00ff41; }
    .stAlert { background-color: #000; border: 1px solid #ff4b4b; color: #ff4b4b; border-radius: 0px; }
    /* Dataframe Styling */
    .stDataFrame { border: 1px solid #3e424b; border-radius: 5px; }
    </style>
    \"\"\", unsafe_allow_html=True)

@st.cache_data
def load_data():
    with open('data_processed_events.json', 'r') as f:
        df = pd.DataFrame(json.load(f))
    df['event_datetime_utc'] = pd.to_datetime(df['event_datetime_utc'])
    return df.sort_values('event_datetime_utc', ascending=False)

df = load_data()

# 2. DRILL-DOWN CAPABILITY (Sidebar Filters)
with st.sidebar:
    st.title("🔍 DRILL-DOWN")
    st.markdown("---")
    loc_select = st.multiselect("SECTOR LOCK", df['location_text'].unique(), default=df['location_text'].unique())
    source_select = st.multiselect("SOURCE FILTER", df['source_name'].unique(), default=df['source_name'].unique())

    # Filter the data based on user selection
    f_df = df[(df['location_text'].isin(loc_select)) & (df['source_name'].isin(source_select))]

    st.markdown("---")
    st.code("MODE: ANALYST_OVERRIDE\\nSTATUS: ENCRYPTED", language="bash")
    st.image("https://img.icons8.com/ios-filled/50/00ff41/marker.png", width=30)

# 3. EXECUTIVE SUMMARY (Header KPIs)
st.title("🛰️ OMNI-RECON | STRATEGIC COMMAND")
st.markdown(f"**POSTURE:** {'CRITICAL' if f_df['severity_score'].mean() > 6 else 'MONITORING'} | **LAST SYNC:** {datetime.now().strftime('%H:%M:%S')} UTC")

c1, c2, c3, c4 = st.columns(4)
c1.metric("THREAT LEVEL", f"{f_df['severity_score'].mean():.1f}", delta="SIGNAL SURGE")
c2.metric("SIGNAL COUNT", len(f_df), delta=f"+{random.randint(1,5)}")
c3.metric("CONFIDENCE", "94%", delta="STABLE")
c4.metric("VERIFIED", f"{len(f_df[f_df['information_status']=='verified'])}")

# 4. TREND VIEW (Escalation Signaling)
st.subheader("📈 INTENSITY TREND (TEMPORAL ANALYSIS)")
trend_df = f_df.groupby(f_df['event_datetime_utc'].dt.date)['severity_score'].mean().reset_index()
fig = px.area(trend_df, x='event_datetime_utc', y='severity_score',
              color_discrete_sequence=['#00ff41'], template="plotly_dark")
fig.update_layout(plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', yaxis_title="AVG SEVERITY")
st.plotly_chart(fig, use_container_width=True)

# 5. LOCATION / MAP VIEW
st.subheader("📍 GEOSPATIAL KINETIC OVERLAY")
COORDS = {
    "Tehran": [35.68, 51.38], "Shiraz": [29.59, 52.58],
    "Jerusalem": [31.76, 35.21], "Tel Aviv": [32.08, 34.78],
    "Isfahan": [32.65, 51.66], "Haifa": [32.79, 34.98]
}
f_df['lat'] = f_df['location_text'].map(lambda x: COORDS.get(x, [None])[0])
f_df['lon'] = f_df['location_text'].map(lambda x: COORDS.get(x, [None, None])[1])
st.map(f_df.dropna(subset=['lat', 'lon'])[['lat', 'lon']], zoom=4)

# 6. EVENT FEED & SOURCE VISIBILITY
st.subheader("📋 RECONNAISSANCE LOGS (PROVENANCE)")
# Custom styling for the feed to highlight severity
st.dataframe(f_df[['event_datetime_utc', 'source_name', 'location_text', 'claim_text', 'severity_score', 'information_status']],
             use_container_width=True)

# 7. GEN-AI STRATEGIC SUMMARY
with st.expander("🤖 AI_INFERENCE_ENGINE", expanded=True):
    st.write(f"**ASSESSMENT:** Kinetic activity in {', '.join(loc_select[:2])} sectors indicates a multi-vector surge. "
             f"Source correlation from {', '.join(source_select[:2])} confirms severity spikes.")
""")